In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_ollama import ChatOllama
from langgraph.checkpoint.memory import InMemorySaver


In [3]:
llm = ChatOllama(model="llama3.2")


In [4]:
class JokeState(TypedDict): 
    topic: str 
    joke: str 
    explanation: str

In [5]:
def generate_joke(state: JokeState): 
    prompt = f"generate a joke on the topic {state['topic']}"
    response = llm.invoke(prompt).content


    return{'joke': response}

In [6]:
def generate_explanation(state: JokeState): 
    prompt = f"Write an explanation for the given joke = {state['joke']}"
    response = llm.invoke(prompt).content
    return {'explanation': response}

In [ ]:
graph = StateGraph(JokeState)

#node
graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_explanation", generate_explanation)

#edges 
graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_explanation")
graph.add_edge("generate_explanation", END)


checkpointers = InMemorySaver()

workflow = graph.compile()

In [10]:
inital_state = {
    "topic": "Politics"
}

workflow.invoke(inital_state)

{'topic': 'Politics',
 'joke': 'Why did the politician bring a ladder to the debate?\n\nBecause he wanted to take his argument to a higher level!',
 'explanation': 'A clever joke! The joke relies on a play on words, using the multiple meanings of the word "higher" to create a pun.\n\nIn this joke, the politician brings a ladder to the debate, which could be interpreted as a literal tool to elevate himself or his argument. However, the punchline "to take his argument to a higher level" is a clever double entendre. It can be taken literally, as the ladder is meant to physically raise him, but it can also be interpreted figuratively, as the politician is trying to elevate his argument\'s level of complexity, importance, or persuasiveness.\n\nThe joke relies on the audience being familiar with the common phrase "take it to a higher level," which is often used to describe someone taking a complex or nuanced argument. By using the word "higher" in a literal sense, the joke creates a unexpect